# Bounding-box matching

Match predictions to annotations using confidence-ordered greedy assignment, inclusive VOC `xyxy` coordinates, and IoU ≥ 0.5.

## Load the data

Place the downloaded files in `data/`, or set `GFLD_DATA_DIR` to their location. Dataset details are provided in the paper.

In [ ]:
%matplotlib inline
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gfld import *

# Works when Jupyter starts in the repository root or notebooks directory.
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else Path.cwd().parent
DATA_DIR = example_directory(ROOT)  # GFLD_DATA_DIR or repository data/
data = load_example(DATA_DIR)
print({k: v["images"] for k, v in data["metadata"]["splits"].items()})

## Inspect one image

Each prediction has a box, detection confidence, and three class scores. A ground-truth object can be matched only once; duplicate detections remain unmatched.

In [ ]:
prediction = next(p for p in data["internal"]
                  if len(data["internal_annotations"][p.image_id].labels) > 0)
annotation = data["internal_annotations"][prediction.image_id]
matches = match_boxes(prediction, annotation, mode="spatial")
pd.DataFrame({"confidence": prediction.scores,
              "top1": [CLASSES[i] for i in prediction.class_scores.argmax(axis=1)],
              "matched_gt_index": matches})

## View the assignments

Green boxes show annotations. Dashed prediction boxes are blue when matched and red when unmatched.

In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
fig, ax = plt.subplots(figsize=(9, 5))
for j, b in enumerate(annotation.boxes):
    ax.add_patch(Rectangle(b[:2], b[2]-b[0]+1, b[3]-b[1]+1, fill=False, color="green", lw=2))
    ax.text(b[0], b[1]-5, f"GT {j}", color="green")
legend = [Line2D([0], [0], color="green", label="Ground truth")]
for j, b in enumerate(prediction.boxes):
    color = "tab:blue" if matches[j] >= 0 else "tab:red"
    ax.add_patch(Rectangle(b[:2], b[2]-b[0]+1, b[3]-b[1]+1, fill=False, color=color, linestyle="--"))
    status = f"matched GT {matches[j]}" if matches[j] >= 0 else "unmatched"
    legend.append(Line2D([0], [0], color=color, linestyle="--",
                        label=f"P{j}: {prediction.scores[j]:.2f}, {status}"))
ax.autoscale_view(); ax.invert_yaxis(); ax.set_aspect("equal")
ax.set(title=prediction.image_id, xlabel="x (pixels)", ylabel="y (pixels)")
ax.legend(handles=legend, loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
plt.tight_layout(); plt.show()

## Compare matching rules

CP calibration uses spatial-only matching. Filtering first restricts eligible annotations to classes in the predicted label set. The threshold below is illustrative; notebook 02 calibrates it.

In [ ]:
illustrative_sets = LabelSets(threshold=0.01, alpha=0.01, matched_pairs=0).predict(prediction.class_scores)
pd.DataFrame({
    "spatial": matches,
    "top1_label": match_boxes(prediction, annotation, mode="label"),
    "label_set": match_boxes(prediction, annotation, mode="set", label_sets=illustrative_sets),
})

Equal-confidence ties preserve prediction order. Equal-IoU ties preserve annotation order. Empty label sets have no eligible annotations. Proceed to [conformal prediction](02_conformal_prediction.ipynb).